In [ ]:
# !pip install mamba-ssm tilelang quack-kernels --no-build-isolation
import os
import torch
import torchaudio
from IPython.display import display, Audio
import matplotlib.pyplot as plt

In [ ]:
from mamba_ssm import Mamba
import torch
import torch.nn as nn
import torch.nn.functional as F

class MambaDeflationaryExtractor(nn.Module):
    def __init__(self, d_model=256, d_state=16, d_conv=4, expand=2, num_blocks=4):
        super().__init__()
        
        # Encoder: Audio waveform to latent features (1D Conv)
        self.encoder = nn.Conv1d(in_channels=1, out_channels=d_model, kernel_size=16, stride=8, padding=4)
        
        # Backbone: Stacked Mamba Blocks (Ultra-efficient sequential processing)
        self.mamba_blocks = nn.ModuleList([
            Mamba(
                d_model=d_model, # Model dimension
                d_state=d_state, # State dimension (controls memory compression)
                d_conv=d_conv,   # Local convolution width
                expand=expand    # Block expansion factor
            ) for _ in range(num_blocks)
        ])
        
        # Dual Decoders: 
        # Decoder A isolates the dominant speaker
        self.speaker_decoder = nn.ConvTranspose1d(in_channels=d_model, out_channels=1, kernel_size=16, stride=8, padding=4)
        # Decoder B outputs the rest of the mixture
        self.residual_decoder = nn.ConvTranspose1d(in_channels=d_model, out_channels=1, kernel_size=16, stride=8, padding=4)

    def forward(self, x):
        """
        x shape: (batch_size, 1, sequence_length)
        """
        # Encode
        encoded = self.encoder(x) # Shape: (batch, d_model, seq_len_latent)
        
        # Mamba expects shape (batch, seq_len, d_model), so we transpose
        latent = encoded.transpose(1, 2)
        
        # Pass through Mamba backbone
        for block in self.mamba_blocks:
            latent = block(latent)
            
        # Transpose back for decoding
        latent = latent.transpose(1, 2) # Shape: (batch, d_model, seq_len_latent)
        
        # Decode into two separate audio signals
        est_speaker = self.speaker_decoder(latent)
        est_residual = self.residual_decoder(latent)
        
        # Squeeze out the channel dimension for loss calculation
        return est_speaker.squeeze(1), est_residual.squeeze(1)

In [ ]:
def test_single_pass_separation(mixture_path, checkpoint_path, out_dir, max_speakers=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on {device}...\n")
    
    # 1. Initialize and Load Model
    model = MambaDeflationaryExtractor(d_model=256, d_state=16, d_conv=4, expand=2, num_blocks=4)
    
    if not os.path.exists(checkpoint_path):
        print(f" Weights not found at {checkpoint_path}")
        return
        
    state_dict = torch.load(checkpoint_path, map_location=device)
    clean_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(clean_state_dict)
    model.to(device)
    model.eval()
    
    # 2. Load and Format Audio
    mixture, sr = torchaudio.load(mixture_path)
    
    # Convert to mono if it's stereo
    if mixture.shape[0] > 1:
        mixture = mixture.mean(dim=0, keepdim=True)
        
    # Model expects shape: (batch, channels, seq_len)
    current_signal = mixture.unsqueeze(0).to(device) 
    
    os.makedirs(out_dir, exist_ok=True)
    print(f" Processing mixture: {mixture_path}")
    
    # 3. Deflationary Extraction Loop
    with torch.no_grad():
        for i in range(1, max_speakers + 1):
            # est_speaker shape: (batch, seq_len)
            est_speaker, est_residual = model(current_signal)
            
            # Save the isolated speaker
            speaker_audio = est_speaker.cpu() # shape is (1, seq_len)
            out_path = os.path.join(out_dir, f"separated_speaker_{i}.wav")
            torchaudio.save(out_path, speaker_audio, sr)
            print(f" Extracted Speaker {i} -> saved to {out_path}")
            
            # The residual becomes the input for the next iteration
            # Must add channel dimension back: (batch, seq_len) -> (batch, 1, seq_len)
            current_signal = est_residual.unsqueeze(1)

    print("\n Separation complete! ")

In [ ]:
# Paths 
MIXTURE_FILE = "../demo_audio/5_speaker_mixture.wav" # replace 5 with 4,3,2 for more
WEIGHTS_FILE = "../weights/mamba_dexformer_epoch_115.pth"
OUTPUT_DIR = "../demo_audio/separated_outputs/"

# Trigger the single-pass split
# Make sure you actually have a test file at that location before running
if os.path.exists(MIXTURE_FILE):
    test_single_pass_separation(
        mixture_path=MIXTURE_FILE,
        checkpoint_path=WEIGHTS_FILE,
        out_dir=OUTPUT_DIR,
        max_speakers=5
    )
else:
    print(f"Please add a test mixture wav file at {MIXTURE_FILE}")